# RodPrep — exploration

Notebook d'exploration de l'étape 1 : extraction du récap Excel ROD et construction de la table hôtel.

Objectif : visualiser les entrées, les étapes intermédiaires et remplir `../Output/`.

In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

ROOT = Path.cwd().resolve()
while ROOT.name != "RodPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Entrée — récap Excel et registre identité

In [2]:
from rod_ia.config.settings import get_settings
from rod_ia.domain.repositories.identity_registry import HotelIdentityRegistry
from rod_prep.prep import RodPrep

settings = get_settings(PROJECT)
prep = RodPrep(INPUT_DIR, OUTPUT_DIR, settings.identity_registry_path)

recap_path = prep.seed_input_from_sources()
print("Fichier récap :", recap_path)

registry = HotelIdentityRegistry(settings.identity_registry_path)
registry_df = pd.DataFrame([r.to_dict() for r in registry.all_records()])
print(f"Registre identité : {len(registry_df)} hôtels")
display(registry_df.head(3))
registry_df[["hotel_id", "name_ventes", "brand", "city", "nb_chambres"]].head(10)

Fichier récap : /media/laghmari/ssd-data/dev/hotels/prepare/RodPrep/Input/recapitulatif_rod.xlsx
Registre identité : 8 hôtels


,hotel_id,brand,city,name_display,name_ventes,name_rod,aliases,lat_canonical,lon_canonical,geo_source,lat_rod,lon_rod,lat_nominatim,lon_nominatim,has_sales,has_rod,nb_chambres
0,ibis-budget-nice,IBIS BUDGET,Nice,Ibis budget Nice Californie,Ibis budget Nice,Nice Californie,"[Ibis Budget Nice, IBIS BUDGET Nice]",43.710000,7.260000,nominatim,None,None,43.689258,7.240379,True,True,129.0
1,ibis-budget-strasbourg,IBIS BUDGET,Strasbourg,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,Strasbourg République,"[Ibis Budget Strasbourg, Ibis budget Strasbourg]",NaN,NaN,None,None,None,NaN,NaN,True,True,97.0
2,ibis-styles-roissy-cdg,IBIS STYLES,Roissy,Ibis Styles Roissy CDG,None,Roissy CDG,[],49.007078,2.520403,nominatim,None,None,49.007078,2.520403,False,True,309.0


,hotel_id,name_ventes,brand,city,nb_chambres
0,ibis-budget-nice,Ibis budget Nice,IBIS BUDGET,Nice,129.0
1,ibis-budget-strasbourg,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0
2,ibis-styles-roissy-cdg,None,IBIS STYLES,Roissy,309.0
3,novotel-megeve,Novotel Megève Mont-Blanc,NOVOTEL,Megève,572.0
4,novotel-paris-tour-eiffel,Novotel Paris Tour Eiffel,NOVOTEL,Paris,764.0
5,mercure-montmartre,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,305.0
6,mercure-boulogne,None,MERCURE,Boulogne-Billancourt,191.0
7,novotel-porte-italie,Novotel Porte d'Italie,NOVOTEL,Paris,NaN


## 2. Extraction longue — une ligne par variable × hôtel

In [3]:
from rod_ia.domain.services.rod_recap_extractor import RodRecapExtractor

extractor = RodRecapExtractor(
    recap_path=recap_path,
    identity_registry=registry,
    output_path=OUTPUT_DIR / "rod_recap",
)

long_df = extractor.extract_long()
print(f"Format long : {long_df.shape[0]} lignes × {long_df.shape[1]} colonnes")
long_df.head(12)

Format long : 938 lignes × 9 colonnes


,hotel_id,recap_column,row,etape,sous_etape,data_label,field_key,field_type_hint,raw_value
0,ibis-budget-nice,NICE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H2075
1,ibis-budget-strasbourg,STRASBOURG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB6A3
2,ibis-styles-roissy-cdg,PARIS CDG,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0815
3,novotel-megeve,MEGEVE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,HB5I0
4,novotel-paris-tour-eiffel,TOUR EIFFEL,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H3546
5,mercure-montmartre,MONTMARTRE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H0373
6,mercure-boulogne,BOULOGNE,4,0 - PAGE DE CONNEXION,ID,CODE H,0_page_de_connexion_id_code_h,categorical,H6188
7,ibis-budget-nice,NICE,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET NICE CALIFORNIE
8,ibis-budget-strasbourg,STRASBOURG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS BUDGET STRASBOURG REPUBLIQUE
9,ibis-styles-roissy-cdg,PARIS CDG,5,0 - PAGE DE CONNEXION,ID,NOM DE L'HOTEL,0_page_de_connexion_id_nom_de_l_hotel,categorical,IBIS STYLES ROISSY CDG


## 3. Format wide — features `d_recap_*` par hôtel

In [4]:
wide_df = extractor.extract_wide()
print(f"Format wide : {wide_df.shape[0]} hôtels × {wide_df.shape[1]} colonnes")
wide_df.head()

Format wide : 7 hôtels × 90 colonnes


,hotel_id,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_latitude,d_recap_0_page_de_connexion_localisation_geo_longitude,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_a_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_a_cafe,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_refrigeree,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygiene,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pret_a_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosmetiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_corner_autre_emplacement_dispo_metres_carres_estimation,d_recap_corner_autre_emplacement_dispo_metres_lineaires_estimation,d_recap_corner_autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,d_recap_corner_corner_de_vente_actuel_metres_carres_estimation,d_recap_corner_corner_de_vente_actuel_metres_lineaires_estimation,d_recap_corner_corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,d_recap_corner_equipements_disponibles_alimentation_electrique_r124,d_recap_corner_equipements_disponibles_internet_filaire_prise_rj45_r123,d_recap_corner_equipements_disponibles_videosurveillance_r125,d_recap_corner_equipements_disponibles_wifi_r122,d_recap_corner_votre_corner_actuel_offre_f_b_caisse_code_barres,d_recap_corner_votre_corner_actuel_offre_f_b_distributeur_auto,d_recap_corner_votre_corner_actuel_offre_f_b_frigo_connecte,d_recap_corner_votre_corner_actuel_offre_f_b_hotel_staff,d_recap_corner_votre_corner_actuel_offre_f_b_liste_des_produits_f_b,d_recap_corner_votre_corner_actuel_offre_f_b_reception,d_recap_corner_votre_corner_ac

## 4. Table de liaison `hotel_lookup`

In [21]:
hotel_lookup = prep.run()  # persiste aussi rod_features + hotel_lookup
print(f"hotel_lookup : {hotel_lookup.shape}")
hotel_lookup.head()

hotel_lookup : (8, 95)


,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,nb_chambres,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_1,d_recap_0_page_de_connexion_localisation_geo_adresse_postale_2,d_recap_0_page_de_connexion_localisation_geo_code_postal,d_recap_0_page_de_connexion_localisation_geo_ville,d_recap_2_services_equipements_dispo_dans_le_lobby_assises,d_recap_2_services_equipements_dispo_dans_le_lobby_bouilloire,d_recap_2_services_equipements_dispo_dans_le_lobby_fontaine_a_eau,d_recap_2_services_equipements_dispo_dans_le_lobby_machine_a_cafe,d_recap_2_services_equipements_dispo_dans_le_lobby_micro_ondes,d_recap_2_services_equipements_dispo_dans_le_lobby_vitrine_refrigeree,d_recap_2_services_equipements_f_b_bar,d_recap_2_services_equipements_f_b_horaires_d_ouverture,d_recap_2_services_equipements_f_b_horaires_d_ouverture_r43,d_recap_2_services_equipements_f_b_jours_d_ouverture,d_recap_2_services_equipements_f_b_jours_d_ouverture_r44,d_recap_2_services_equipements_f_b_minibar,d_recap_2_services_equipements_f_b_mois_d_ouverture,d_recap_2_services_equipements_f_b_mois_d_ouverture_r45,d_recap_2_services_equipements_f_b_restaurant,d_recap_2_services_equipements_f_b_room_service,d_recap_2_services_equipements_non_f_b_piscine,d_recap_2_services_equipements_non_f_b_salle_de_sport,d_recap_2_services_equipements_non_f_b_salles_de_reunion,d_recap_2_services_equipements_non_f_b_spa,d_recap_3_profil_de_vos_clients_affaires_affaires_pct,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_boissons_non_alcoolisees,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_epicerie_fine,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sales_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_frais,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_f_b_produits_sucres_secs,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_accessoires,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_articles_pour_enfants,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_hygiene,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_pret_a_porter,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_cosmetiques,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_produits_sos,d_recap_3_profil_de_vos_clients_besoins_de_vos_clients_non_f_b_souvenirs,d_recap_3_profil_de_vos_clients_loisirs_loisirs_pct,d_recap_3_profil_de_vos_clients_loisirs_top_1_amis,d_recap_3_profil_de_vos_clients_loisirs_top_1_couples,d_recap_3_profil_de_vos_clients_loisirs_top_1_familles,d_recap_3_profil_de_vos_clients_national_vs_inter_international_pct,d_recap_3_profil_de_vos_clients_national_vs_inter_national_pct,d_recap_corner_autre_emplacement_dispo_metres_carres_estimation,d_recap_corner_autre_emplacement_dispo_metres_lineaires_estimation,d_recap_corner_autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,d_recap_corner_corner_de_vente_actuel_metres_carres_estimation,d_recap_corner_corner_de_vente_actuel_metres_lineaires_estimation,d_recap_corner_corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,d_recap_corner_equipements_disponibles_alimentation_electrique_r124,d_recap_corner_equipements_disponibles_internet_filaire_prise_rj45_r123,d_recap_corner_equipements_disponibles_videosurveillance_r125,d_recap_corner_equipements_disponibles_wifi_r122,d_recap_corner_votre_corner_actuel_offre_f_b_caisse_code_barres,d_recap_corner_votre_corner_actuel_offre_f_b_distributeur_auto,d_recap_corner_votre_corner_actuel_offre_f_b_frigo_connecte,d_recap_corner_votre_corner_actuel_offre_f_b_hotel_staff,d_recap_corner_votre_corner_actuel_offre_f_b_liste_des_produits_f_b,d_recap_corner_votre_corner_actuel_offre_f_b_reception,d_recap_corner_votre_corner_actuel_offre_f_b_snacking_comptoir,d_recap_corner_vot

## 5. Aperçu colonnes récap retenues

In [6]:
recap_cols = [c for c in hotel_lookup.columns if str(c).startswith("d_recap_")]
print(f"{len(recap_cols)} colonnes d_recap_")
if recap_cols:
    hotel_lookup[["hotel_code", "nom_hotel"] + recap_cols[:8]].head()

86 colonnes d_recap_


## 6. Entrée MeteoPrep / ProximityPrep

In [7]:
meteo_input = prep.to_meteo_input()
meteo_input

,hotel_code,hotel_name,hotel_brand,hotel_city,hotel_lat,hotel_lon
0,H2075,Ibis budget Nice Californie,IBIS BUDGET,Nice,43.689186,7.240512
1,HB6A3,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,48.591522,7.754599
2,H0815,Ibis Styles Roissy CDG,IBIS STYLES,Roissy,49.006733,2.519843
3,H6188,Mercure Paris Boulogne,MERCURE,Boulogne-Billancourt,48.833827,2.256274
4,H0373,Mercure Paris Montmartre Sacré-Cœur,MERCURE,Paris,48.885048,2.329923
5,HB5I0,Novotel Megève Mont-Blanc,NOVOTEL,Megève,45.859165,6.619055
6,H3546,Novotel Paris Centre Tour Eiffel,NOVOTEL,Paris,48.849778,2.282836


## 7. Fichiers produits dans Output/

In [8]:
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)

hotel_lookup.csv
hotel_lookup.parquet
rod_features.csv
rod_features.parquet
rod_recap.long.csv
rod_recap.schema.json
rod_recap.wide.csv


In [17]:
# df = hotel_lookup

In [22]:
unwanted_prefixes= [
    "d_recap_0_page_de_connexion_localisation_geo_",
    "d_recap_2_services_equipements_",
    "d_recap_3_profil_de_vos_clients_affaires_" ,
    "d_recap_3_profil_de_vos_clients_affaires_",
    "d_recap_3_profil_de_vos_clients_",
    "d_recap_corner_",
    "d_recap_de_controle_parametres_",
    "d_recap_generales_donnees_admin_",
    "d_recap_generales_donnees_chiffrees_",
    ""
]


hotel_lookup.columns = hotel_lookup.columns.str.replace(
    "|".join(unwanted_prefixes), 
    "", 
    regex=True
).str.strip().str.replace(r'_+', '_', regex=True).str.strip('_')

In [29]:
hotel_lookup

,hotel_code,hotel_name,nom_hotel,hotel_brand,hotel_city,nb_chambres,adresse_postale_1,adresse_postale_2,code_postal,ville,dispo_dans_le_lobby_assises,dispo_dans_le_lobby_bouilloire,dispo_dans_le_lobby_fontaine_a_eau,dispo_dans_le_lobby_machine_a_cafe,dispo_dans_le_lobby_micro_ondes,dispo_dans_le_lobby_vitrine_refrigeree,f_b_bar,f_b_horaires_d_ouverture,f_b_horaires_d_ouverture_r43,f_b_jours_d_ouverture,f_b_jours_d_ouverture_r44,f_b_minibar,f_b_mois_d_ouverture,f_b_mois_d_ouverture_r45,f_b_restaurant,f_b_room_service,non_f_b_piscine,non_f_b_salle_de_sport,non_f_b_salles_de_reunion,non_f_b_spa,affaires_pct,besoins_de_vos_clients_f_b_boissons_alcoolisees,besoins_de_vos_clients_f_b_boissons_non_alcoolisees,besoins_de_vos_clients_f_b_epicerie_fine,besoins_de_vos_clients_f_b_produits_sales_frais,besoins_de_vos_clients_f_b_produits_sales_secs,besoins_de_vos_clients_f_b_produits_sucres_frais,besoins_de_vos_clients_f_b_produits_sucres_secs,besoins_de_vos_clients_non_f_b_accessoires,besoins_de_vos_clients_non_f_b_articles_pour_enfants,besoins_de_vos_clients_non_f_b_hygiene,besoins_de_vos_clients_non_f_b_pret_a_porter,besoins_de_vos_clients_non_f_b_produits_cosmetiques,besoins_de_vos_clients_non_f_b_produits_sos,besoins_de_vos_clients_non_f_b_souvenirs,loisirs_loisirs_pct,loisirs_top_1_amis,loisirs_top_1_couples,loisirs_top_1_familles,national_vs_inter_international_pct,national_vs_inter_national_pct,autre_emplacement_dispo_metres_carres_estimation,autre_emplacement_dispo_metres_lineaires_estimation,autre_emplacement_dispo_xpct_de_mes_clients_passent_devant,corner_de_vente_actuel_metres_carres_estimation,corner_de_vente_actuel_metres_lineaires_estimation,corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un,equipements_disponibles_alimentation_electrique_r124,equipements_disponibles_internet_filaire_prise_rj45_r123,equipements_disponibles_videosurveillance_r125,equipements_disponibles_wifi_r122,votre_corner_actuel_offre_f_b_caisse_code_barres,votre_corner_actuel_offre_f_b_distributeur_auto,votre_corner_actuel_offre_f_b_frigo_connecte,votre_corner_actuel_offre_f_b_hotel_staff,votre_corner_actuel_offre_f_b_liste_des_produits_f_b,votre_corner_actuel_offre_f_b_reception,votre_corner_actuel_offre_f_b_snacking_comptoir,votre_corner_actuel_offre_non_f_b_armoire_connectee,votre_corner_actuel_offre_non_f_b_caisse_code_barres,votre_corner_actuel_offre_non_f_b_distributeur_auto,votre_corner_actuel_offre_non_f_b_hotel_staff,votre_corner_actuel_offre_non_f_b_liste_des_produits_non_f,votre_corner_actuel_offre_non_f_b_reception,metres_lineaires_dedies_a_v,moyen_de_guests_par_chambre,nb_de_chambres,to_annuel_moyen,contrat_signe_annee,contrat_type,derniere_reno_hotel,derniere_reno_lobby,dom_dof,marque,nb_de_chambres,pms,proprietaire,to_annuel,to_le_plus_bas_mois,to_le_plus_bas_taux,to_le_plus_haut_mois,to_le_plus_haut_taux,hotel_lat,hotel_lon,hotel_geo_source
0,H2075,Ibis budget Nice Californie,Ibis budget Nice,IBIS BUDGET,Nice,129.0,58-60 AVENUE DE LA CALIFORNIE,NaN,06200,NICE,1.0,NaN,1.0,1.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,4.0,0.0,NaN,NaN,1.0,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,1.0,NaN,NaN,NaN,NaN,0.0,1.0,0.0,NaN,1.0,0.0,0.0,0.0,0.0,1.0,NaN,0.0,0.0,6.0,NaN,129.0,NaN,NaN,FRANCHISE,NaN,NaN,Marion BOROT,IBIS BUDGET,129.0,FOLS,FAMILLE FARINES,NaN,NaN,NaN,NaN,NaN,43.689186,7.240512,recap
1,HB6A3,Ibis budget Strasbourg Centre République,Ibis budget Strasbourg Centre République,IBIS BUDGET,Strasbourg,97.0,23A RUE OBERLIN,NaN,67000,STRASBOURG,1.0,0.0,1.0,1.0,0.0,0.0,0.0,16h - 00h,NaN,MARDI AU SAMEDI,NaN,1.0,TOUS,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.4,NaN,1.0,NaN,1.0,1.0,1.0,1.0,NaN,NaN,1.0,NaN,NaN,1.0,NaN,0.6,1.0,NaN,NaN,0.4,0.6,3.0,3.0,100.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,2.0,1.5,97.0,0.70,2020.0,FRANCHISE,2025.0,2025.0,Christophe RUQUEBOEUCHE,IBIS BUDGET,97.0,OPERA,MILLION - CHOREIN,0.70,JANVIER,0.60,DECEMBRE,0.80,48.591522,7.754599,r

In [30]:
map_cols = {'hotel_code': 'hotel_code',
 'hotel_name': 'hotel_name',
#  'nom_hotel': 'nom_hotel',
 'hotel_brand': 'hotel_brand',
 'hotel_city': 'hotel_city',
 'adresse_postale_1': 'hotel_adresse_postale_1',
 'adresse_postale_2': 'hotel_adresse_postale_2',
 'code_postal': 'hotel_code_postal',
#  'ville': 'ville',

 'hotel_lat': 'hotel_lat',
 'hotel_lon': 'hotel_lon',



#   'marque': 'hotel_marque',
 'contrat_signe_annee': 'hotel_contrat_signe_annee',
 'contrat_type': 'hotel_contrat_type',
 'derniere_reno_hotel': 'hotel_derniere_reno',
 'derniere_reno_lobby': 'hotel_lobby_derniere_reno',
#  'dom_dof': 'dom_dof',
#  'pms': 'hotel_pms',
#  'proprietaire': 'hotel_proprietaire',
 'nb_chambres': 'hotel_nb_chambres',
 'to_annuel': 'hotel_to_annuel',
 # 'nb_de_chambres': 'hotel_nb_de_chambres',
#  'moyen_de_guests_par_chambre': 'hotel_moyen_de_guests_par_chambre',
#  'to_annuel_moyen': 'hotel_to_annuel_moyen',

#  'to_le_plus_bas_mois': 'hotel_to_le_plus_bas_mois',
 'to_le_plus_bas_taux': 'hotel_to_le_plus_bas_taux',
#  'to_le_plus_haut_mois': 'hotel_to_le_plus_haut_mois',
 'to_le_plus_haut_taux': 'hotel_to_le_plus_haut_taux',





 'dispo_dans_le_lobby_assises': 'hotel_dispo_dans_lobby_assises',
 'dispo_dans_le_lobby_bouilloire': 'hotel_dispo_dans_lobby_bouilloire',
 'dispo_dans_le_lobby_fontaine_a_eau': 'hotel_dispo_dans_lobby_fontaine_a_eau',
 'dispo_dans_le_lobby_machine_a_cafe': 'hotel_dispo_dans_lobby_machine_a_cafe',
 'dispo_dans_le_lobby_micro_ondes': 'hotel_dispo_dans_lobby_micro_ondes',
 'dispo_dans_le_lobby_vitrine_refrigeree': 'hotel_dispo_dans_lobby_vitrine_refrigeree',
 
 'f_b_bar': 'hotel_f_b_bar',
#  'f_b_horaires_d_ouverture': 'f_b_horaires_d_ouverture',
#  'f_b_horaires_d_ouverture_r43': 'f_b_horaires_d_ouverture_r43',
#  'f_b_jours_d_ouverture': 'f_b_jours_d_ouverture',
#  'f_b_jours_d_ouverture_r44': 'f_b_jours_d_ouverture_r44',
 'f_b_minibar': 'hotel_f_b_minibar',
#  'f_b_mois_d_ouverture': 'f_b_mois_d_ouverture',
#  'f_b_mois_d_ouverture_r45': 'f_b_mois_d_ouverture_r45',
 'f_b_restaurant': 'hotel_f_b_restaurant',
 'f_b_room_service': 'hotel_f_b_room_service',

 'non_f_b_piscine': 'hotel_non_f_b_piscine',
 'non_f_b_salle_de_sport': 'hotel_non_f_b_salle_de_sport',
 'non_f_b_salles_de_reunion': 'hotel_non_f_b_salles_de_reunion',
 'non_f_b_spa': 'hotel_non_f_b_spa',



#  'besoins_de_vos_clients_f_b_boissons_alcoolisees': 'besoins_f_b_boissons_alcoolisees',
#  'besoins_de_vos_clients_f_b_boissons_non_alcoolisees': 'besoins_f_b_boissons_non_alcoolisees',
#  'besoins_de_vos_clients_f_b_epicerie_fine': 'besoins_f_b_epicerie_fine',
#  'besoins_de_vos_clients_f_b_produits_sales_frais': 'besoins_f_b_produits_sales_frais',
#  'besoins_de_vos_clients_f_b_produits_sales_secs': 'besoins_f_b_produits_sales_secs',
#  'besoins_de_vos_clients_f_b_produits_sucres_frais': 'besoins_f_b_produits_sucres_frais',
#  'besoins_de_vos_clients_f_b_produits_sucres_secs': 'besoins_f_b_produits_sucres_secs',

#  'besoins_de_vos_clients_non_f_b_accessoires': 'besoins_non_f_b_accessoires',
#  'besoins_de_vos_clients_non_f_b_articles_pour_enfants': 'besoins_non_f_b_articles_pour_enfants',
#  'besoins_de_vos_clients_non_f_b_hygiene': 'besoins_non_f_b_hygiene',
#  'besoins_de_vos_clients_non_f_b_pret_a_porter': 'besoins_non_f_b_pret_a_porter',
#  'besoins_de_vos_clients_non_f_b_produits_cosmetiques': 'besoins_non_f_b_produits_cosmetiques',
#  'besoins_de_vos_clients_non_f_b_produits_sos': 'besoins_non_f_b_produits_sos',
#  'besoins_de_vos_clients_non_f_b_souvenirs': 'besoins_non_f_b_souvenirs',

 'affaires_pct': 'hotel_affaires_pct',
 'loisirs_loisirs_pct': 'hotel_loisirs_pct',
 'loisirs_top_1_amis': 'hotel_loisirs_top_1_amis',
 'loisirs_top_1_couples': 'hotel_loisirs_top_1_couples',
 'loisirs_top_1_familles': 'hotel_loisirs_top_1_familles',

 'national_vs_inter_international_pct': 'hotel_international_pct',
 'national_vs_inter_national_pct': 'hotel_national_pct',
 
#  'autre_emplacement_dispo_metres_carres_estimation': 'autre_emplacement_dispo_metres_carres_estimation',
#  'autre_emplacement_dispo_metres_lineaires_estimation': 'autre_emplacement_dispo_metres_lineaires_estimation',
#  'autre_emplacement_dispo_xpct_de_mes_clients_passent_devant': 'autre_emplacement_dispo_xpct_de_mes_clients_passent_devant',
  
 'corner_de_vente_actuel_votre_hotel_dispose_t_il_deja_d_un': 'hotel_corner_actuel_existe_deja',
 'corner_de_vente_actuel_metres_lineaires_estimation': 'hotel_corner_de_vente_actuel_metres_lineaires',

 'votre_corner_actuel_offre_f_b_caisse_code_barres': 'hotel_corner_actuel_offre_f_b_caisse_code_barres',
 'votre_corner_actuel_offre_f_b_distributeur_auto': 'hotel_corner_actuel_offre_f_b_distributeur_auto',
 'votre_corner_actuel_offre_f_b_frigo_connecte': 'hotel_corner_actuel_offre_f_b_frigo_connecte',
 'votre_corner_actuel_offre_f_b_reception': 'hotel_corner_actuel_offre_f_b_reception',
 'votre_corner_actuel_offre_f_b_snacking_comptoir': 'hotel_corner_actuel_offre_f_b_snacking_comptoir',
#  'votre_corner_actuel_offre_f_b_hotel_staff': 'hohtel_corner_actuel_offre_f_b_reappro_hotel_staff',

#  'corner_de_vente_actuel_metres_carres_estimation': 'corner_de_vente_actuel_metres_carres_estimation',
 
 

#  'equipements_disponibles_alimentation_electrique_r124': 'equipements_disponibles_alimentation_electrique_r124',
#  'equipements_disponibles_internet_filaire_prise_rj45_r123': 'equipements_disponibles_internet_filaire_prise_rj45_r123',
#  'equipements_disponibles_videosurveillance_r125': 'equipements_disponibles_videosurveillance_r125',
#  'equipements_disponibles_wifi_r122': 'equipements_disponibles_wifi_r122',



#  'votre_corner_actuel_offre_f_b_liste_des_produits_f_b': 'votre_corner_actuel_offre_f_b_liste_des_produits_f_b',
 

 'votre_corner_actuel_offre_non_f_b_armoire_connectee': 'hotel_corner_actuel_offre_non_f_b_armoire_connectee',
 'votre_corner_actuel_offre_non_f_b_caisse_code_barres': 'hotel_corner_actuel_offre_non_f_b_caisse_code_barres',
 'votre_corner_actuel_offre_non_f_b_distributeur_auto': 'hotel_corner_actuel_offre_non_f_b_distributeur_auto',
#  'votre_corner_actuel_offre_non_f_b_hotel_staff': 'hotel_corner_actuel_offre_non_f_b_hotel_staff',
#  'votre_corner_actuel_offre_non_f_b_liste_des_produits_non_f': 'hotel_corner_actuel_offre_non_f_b_liste_des_produits_non_f',
 'votre_corner_actuel_offre_non_f_b_reception': 'hotel_corner_actuel_offre_non_f_b_reception',


 

 'metres_lineaires_dedies_a_v': 'hotel_metres_lineaires_dedies_corner',


#  'hotel_geo_source': 'hotel_geo_source'
 }